# HSV Universality

This notebook demonstrates that Hankel singular values (HSVs) are **universal** — they depend only on the task structure (timescales, number of channels), not on random training seeds or initialization.

**Claim:** For a MIMO SSM trained on a multi-timescale EMA task, the HSV spectrum is deterministic across seeds. Two models trained from different random initializations will converge to the same HSVs (up to numerical precision).

**Why this matters:** If HSVs were seed-dependent, they could not be used as principled circuit energy measures. Universality is the foundation of the entire balanced-realization approach to interpretability.

**Reference:** §2 of the paper; HSV Universality Theorem (Proposition 1).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

import ssm_interpret as ssi

print(f'ssm_interpret {ssi.__version__}')

In [ ]:
# ── Task: 3-channel EMA, synthetic uniform inputs ──────────────────────────
T_SEQ = 32
M = 3
RHOS = [0.99, 0.85, 0.50]
D_STATE = 3
N_TR = 2048
N_TEST = 512
EPOCHS = 20000

def make_ema_data(n, T, rhos, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.uniform(-1, 1, (n, T, len(rhos))).astype(np.float32)
    y = np.zeros_like(u)
    for k, rho in enumerate(rhos):
        s = 0.0
        for t in range(T):
            s = rho * s + (1 - rho) * u[:, t, k]
            y[:, t, k] = s
    return torch.tensor(u), torch.tensor(y)

u_tr, y_tr = make_ema_data(N_TR, T_SEQ, RHOS, seed=0)
u_te, y_te = make_ema_data(N_TEST, T_SEQ, RHOS, seed=1)
print(f'Train: {u_tr.shape}, Test: {u_te.shape}')

In [ ]:
class LinearSSM(nn.Module):
    def __init__(self, m, d, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.A_raw = nn.Parameter(torch.randn(d, d) * 0.1)
        self.B = nn.Parameter(torch.randn(d, m) * 0.1)
        self.C = nn.Parameter(torch.randn(m, d) * 0.1)
        self.d = d

    def _A(self):
        sr = torch.linalg.matrix_norm(self.A_raw, ord=2)
        return self.A_raw / sr.clamp(min=1.0) * 0.99

    def forward(self, u):
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        ys = []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            ys.append(x @ self.C.T)
        return torch.stack(ys, dim=1)

    def get_matrices(self):
        with torch.no_grad():
            return (self._A().numpy().astype(np.float64),
                    self.B.detach().numpy().astype(np.float64),
                    self.C.detach().numpy().astype(np.float64))

def train_model(model, u_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=3e-3)
    sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[8000, 15000], gamma=0.3)
    for _ in range(epochs):
        opt.zero_grad()
        F.mse_loss(model(u_tr), y_tr).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    with torch.no_grad():
        return F.mse_loss(model(u_te), y_te).item()

N_SEEDS = 10
all_sigma = []

print(f'Training {N_SEEDS} models from different random seeds...')
for seed in range(N_SEEDS):
    m = LinearSSM(M, D_STATE, seed=seed)
    test_loss = train_model(m, u_tr, y_tr)
    A, B, C = m.get_matrices()
    _, _, _, _, sigma = ssi.balanced_realization(A, B, C, T=T_SEQ)
    all_sigma.append(np.array(sigma))
    pct = 100 * test_loss / float(y_te.pow(2).mean())
    print(f'  seed={seed}: test_loss={pct:.2f}%  sigma={np.array(sigma).round(4)}')

sigma_arr = np.stack(all_sigma)  # (N_SEEDS, d)
print(f'\nHSV mean across seeds: {sigma_arr.mean(0).round(6)}')
print(f'HSV std  across seeds: {sigma_arr.std(0).round(6)}')
print(f'HSV cv   across seeds: {(sigma_arr.std(0)/(sigma_arr.mean(0)+1e-12)).round(6)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2196F3', '#4CAF50', '#FF9800']

# HSV per seed
ax = axes[0]
for k in range(D_STATE):
    ax.plot(range(N_SEEDS), sigma_arr[:, k], 'o-', color=colors[k],
            label=f'σ_{k} (ρ={RHOS[k]})', markersize=5)
ax.set_xlabel('Training seed')
ax.set_ylabel('Hankel singular value')
ax.set_title('HSV Universality: Same Values Across Seeds')
ax.legend()
ax.set_xticks(range(N_SEEDS))

# Mean ± std bars
ax = axes[1]
ax.bar(range(D_STATE), sigma_arr.mean(0), yerr=sigma_arr.std(0),
       color=colors, edgecolor='k', linewidth=0.8, capsize=6, alpha=0.85)
ax.set_xlabel('Mode index k')
ax.set_ylabel('HSV (mean ± std over seeds)')
ax.set_title(f'HSV Spectrum ({N_SEEDS} seeds)')
ax.set_xticks(range(D_STATE))
for k in range(D_STATE):
    mu, s = sigma_arr.mean(0)[k], sigma_arr.std(0)[k]
    ax.text(k, mu + sigma_arr.std(0)[k] + 0.002,
            f'{mu:.4f}\n±{s:.2e}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('hsv_universality.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: hsv_universality.pdf')

In [ ]:
# Bonus: HSV vs ρ relationship
# Train single-channel models with varying ρ to show HSV scales with timescale
rho_vals = np.linspace(0.1, 0.99, 15)
hsv_vs_rho = []

u1_tr, y1_tr = make_ema_data(N_TR, T_SEQ, [0.5], seed=0)
for rho in rho_vals:
    u_r, y_r = make_ema_data(N_TR, T_SEQ, [rho], seed=0)
    u_r_te, y_r_te = make_ema_data(N_TEST, T_SEQ, [rho], seed=1)
    m = LinearSSM(1, 1, seed=0)
    train_model(m, u_r, y_r, epochs=10000)
    A_r, B_r, C_r = m.get_matrices()
    _, _, _, _, sigma_r = ssi.balanced_realization(A_r, B_r, C_r, T=T_SEQ)
    hsv_vs_rho.append(float(sigma_r[0]))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rho_vals, hsv_vs_rho, 'o-', color='#2196F3')
ax.set_xlabel('EMA decay rate ρ')
ax.set_ylabel('Hankel singular value σ₀')
ax.set_title('HSV Scales with Timescale')
plt.tight_layout()
plt.savefig('hsv_vs_rho.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: hsv_vs_rho.pdf')